# Portfolio Bot — Final (45% Sell at ATH)

**Optimum from grid search:** Sell 45% at each new ATH gives best risk-adjusted returns.

- $20/mo external injection into shared USDT reserve
- BTC: buy $10 + progressive reinvest (1%+2%->50%) on red monthly candle
- BNB: buy $10 + progressive reinvest (1%+1%->40%) on red monthly candle
- Sell 45% of each at their own new ATH close

In [ ]:
import pandas as pd, requests, time, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates
from datetime import datetime, timezone
pd.set_option('display.max_columns', None); pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
def fetch_klines(symbol):
    url = 'https://api.binance.com/api/v3/klines'
    all_k = []; st = 0
    while True:
        p = {'symbol': symbol, 'interval': '1M', 'startTime': st, 'limit': 500}
        d = requests.get(url, params=p).json()
        if not d: break
        all_k.extend(d)
        if len(d) < 500: break
        st = d[-1][0] + 1; time.sleep(0.1)
    return all_k
cols = ['timestamp','open','high','low','close','volume','close_time','quote_vol','trades','taker_buy_base','taker_buy_quote','ignore']
btc_k = fetch_klines('BTCUSDT'); bnb_k = fetch_klines('BNBUSDT')
btc = pd.DataFrame(btc_k, columns=cols); btc['date'] = pd.to_datetime(btc['timestamp'], unit='ms', utc=True)
bnb = pd.DataFrame(bnb_k, columns=cols); bnb['date'] = pd.to_datetime(bnb['timestamp'], unit='ms', utc=True)
for c in ['open','high','low','close','volume']: btc[c] = btc[c].astype(float); bnb[c] = bnb[c].astype(float)
btc = btc[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)
bnb = bnb[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)
start = max(btc['date'].min(), bnb['date'].min())
end = min(btc['date'].max(), bnb['date'].max())
btc = btc[(btc['date'] >= start) & (btc['date'] <= end)].reset_index(drop=True)
bnb = bnb[(bnb['date'] >= start) & (bnb['date'] <= end)].reset_index(drop=True)
print(f'BTC: {btc["date"].min().strftime("%Y-%m")} -> {btc["date"].max().strftime("%Y-%m")} ({len(btc)} months)')
print(f'BNB: {bnb["date"].min().strftime("%Y-%m")} -> {bnb["date"].max().strftime("%Y-%m")} ({len(bnb)} months)')

In [ ]:
SELL_RATIO = 0.45
usdt = 0.0; injected = 0.0
btc_h = 0.0; btc_hi = 0.0; btc_rp = 1.0; btc_inv = 0.0
bnb_h = 0.0; bnb_hi = 0.0; bnb_rp = 1.0; bnb_inv = 0.0
records = []
for i in range(len(btc)):
    br = btc.iloc[i]; nr = bnb.iloc[i]
    bc = br['close']; nc = nr['close']; date = br['date']
    action = []
    usdt += 20.0; injected += 20.0
    # BTC buy on red
    if bc < br['open']:
        ext = usdt * (btc_rp / 100.0) if usdt > 0 else 0.0
        spend = min(10.0 + ext, usdt)
        btc_h += spend / bc; btc_inv += 10.0; usdt -= spend
        action.append(f'BTC buy ${spend:.0f} @ {bc:.0f}')
    # BNB buy on red
    if nc < nr['open']:
        ext = usdt * (bnb_rp / 100.0) if usdt > 0 else 0.0
        spend = min(10.0 + ext, usdt)
        bnb_h += spend / nc; bnb_inv += 10.0; usdt -= spend
        action.append(f'BNB buy ${spend:.0f} @ {nc:.0f}')
    # BTC sell at new ATH
    bp = btc_hi
    if bc > btc_hi: btc_hi = bc
    if btc_h > 0.000001 and bc > bp:
        sell = btc_h * SELL_RATIO
        usdt += sell * bc; btc_h -= sell; btc_rp = 1.0
        action.append(f'BTC sell {SELL_RATIO*100:.0f}% @ {bc:.0f} (-{sell:.6f})')
    elif btc_rp < 50: btc_rp = min(50, btc_rp + 2)
    # BNB sell at new ATH
    np_ = bnb_hi
    if nc > bnb_hi: bnb_hi = nc
    if bnb_h > 0.000001 and nc > np_:
        sell = bnb_h * SELL_RATIO
        usdt += sell * nc; bnb_h -= sell; bnb_rp = 1.0
        action.append(f'BNB sell {SELL_RATIO*100:.0f}% @ {nc:.0f} (-{sell:.6f})')
    elif bnb_rp < 40: bnb_rp = min(40, bnb_rp + 1)
    records.append({'date': date, 'btc_close': bc, 'bnb_close': nc,
        'btc_held': btc_h, 'btc_value': btc_h * bc, 'btc_reinvest': btc_rp,
        'bnb_held': bnb_h, 'bnb_value': bnb_h * nc, 'bnb_reinvest': bnb_rp,
        'usdt_reserve': usdt, 'total_injected': injected,
        'portfolio': btc_h * bc + bnb_h * nc + usdt,
        'action': ' | '.join(action) if action else 'nothing',
        'btc_invested': btc_inv, 'bnb_invested': bnb_inv})
res = pd.DataFrame(records)
print(f'Total months: {len(res)}')
print(f'Months with action: {len(res[res["action"] != "nothing"])}')
buy_mo = len(res[res['action'].str.contains('buy', na=False)])
sell_mo = len(res[res['action'].str.contains('sell', na=False)])
print(f'Buy months: {buy_mo} | Sell months: {sell_mo}')

In [ ]:
final = res.iloc[-1]
total_inv = final['total_injected']
port_val = final['portfolio']
total_pnl = port_val - total_inv
ret_pct = (port_val / total_inv - 1) * 100
eq = res['portfolio']
running_max = eq.cummax()
dd_raw = (running_max - eq) / running_max
max_dd = dd_raw.max() * 100
m = eq.pct_change().dropna()
m = m[m.index >= 12]
sharpe = (m.mean() / m.std()) * np.sqrt(12) if m.std() > 0 else 0
pos = m[m > 0].sum(); neg = m[m < 0].abs().sum()
pf = pos / neg if neg > 0 else float('inf')
ann = (1 + ret_pct/100) ** (12/len(res)) - 1
calmar = ann / (max_dd/100) if max_dd > 0 else 0
btc_ret = (final['btc_value'] / final['btc_invested'] - 1) * 100 if final['btc_invested'] > 0 else 0
bnb_ret = (final['bnb_value'] / final['bnb_invested'] - 1) * 100 if final['bnb_invested'] > 0 else 0
print('='*72)
print('PORTFOLIO BOT — FINAL (45% Sell at ATH)')
print('='*72)
print(f'Period:             {res["date"].min().strftime("%Y-%m")} -> {res["date"].max().strftime("%Y-%m")} ({len(res)} months)')
print(f'External injection: $20/mo = ${total_inv:.0f} total')
print()
print('--- ASSET BREAKDOWN ---')
print(f'BTC invested base:  ${final["btc_invested"]:.0f}  (${final["btc_invested"]/len(res):.2f}/mo)')
print(f'BTC held:           {final["btc_held"]:.8f}  (${final["btc_value"]:.2f} @ {final["btc_close"]:.0f})')
print(f'BTC return (base):  {btc_ret:.1f}%')
print(f'BNB invested base:  ${final["bnb_invested"]:.0f}  (${final["bnb_invested"]/len(res):.2f}/mo)')
print(f'BNB held:           {final["bnb_held"]:.8f}  (${final["bnb_value"]:.2f} @ {final["bnb_close"]:.0f})')
print(f'BNB return (base):  {bnb_ret:.1f}%')
print()
print('--- PORTFOLIO ---')
print(f'USDT reserve:       ${final["usdt_reserve"]:.2f}')
print(f'Portfolio value:    ${port_val:.2f}')
print(f'Total P&L:          ${total_pnl:.2f}')
print(f'Return:             {ret_pct:.2f}%')
print(f'Max drawdown:       {max_dd:.2f}%')
print(f'Sharpe ratio:       {sharpe:.2f}')
print(f'Profit factor:      {pf:.2f}')
print(f'Calmar ratio:       {calmar:.2f}')
print()
print('--- CASHFLOW ---')
print(f'Total injected:     ${total_inv:.0f}')
print(f'In BTC base:        ${final["btc_invested"]:.0f}')
print(f'In BNB base:        ${final["bnb_invested"]:.0f}')
print(f'Reinvested:         ${total_inv - final["btc_invested"] - final["bnb_invested"]:.0f}')
print(f'Reserve remaining:  ${final["usdt_reserve"]:.2f}')
print(f'BTC buys (total):   ${sum(r["btc_invested"] for r in records):.0f}')
print(f'BNB buys (total):   ${sum(r["bnb_invested"] for r in records):.0f}')

In [ ]:
# Chart 1: Full dashboard
fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(5, 2, height_ratios=[3, 1, 1.2, 1.2, 1.2])
ax = fig.add_subplot(gs[0, :])
ax.fill_between(res['date'], res['total_injected'], res['portfolio'],
    where=res['portfolio'] >= res['total_injected'], color='green', alpha=0.12, label='Profit')
ax.fill_between(res['date'], res['portfolio'], res['total_injected'],
    where=res['portfolio'] < res['total_injected'], color='red', alpha=0.12, label='Loss')
ax.plot(res['date'], res['total_injected'], color='gray', linewidth=1.5, linestyle='--', label='Total Injected ($20/mo)')
ax.plot(res['date'], res['portfolio'], color='#2563eb', linewidth=2, label='Portfolio Value')
buys = res[res['action'].str.contains('buy', na=False)]
sells = res[res['action'].str.contains('sell', na=False)]
ax.scatter(buys['date'], buys['portfolio'], color='#16a34a', s=20, marker='^', zorder=5, alpha=0.6, label='Buy')
ax.scatter(sells['date'], sells['portfolio'], color='#dc2626', s=40, marker='v', zorder=5, label='Sell 45%')
ax.set_title('Portfolio Bot — 45% Sell at ATH ($20/mo injection, BTC 1%+2%->50%, BNB 1%+1%->40%)', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=8, ncol=3); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[1, :])
ax.fill_between(res['date'], 0, dd_raw * 100, color='#dc2626', alpha=0.35)
ax.plot(res['date'], dd_raw * 100, color='#dc2626', linewidth=0.8)
ax.axhline(y=max_dd, color='#991b1b', linestyle=':', linewidth=1, label=f'Max DD: {max_dd:.1f}%')
ax.set_ylabel('Drawdown (%)'); ax.set_ylim(bottom=0); ax.legend(loc='lower left', fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
# BTC value
ax = fig.add_subplot(gs[2, 0])
ax.fill_between(res['date'], 0, res['btc_value'], color='#f59e0b', alpha=0.3)
ax.plot(res['date'], res['btc_value'], color='#d97706', linewidth=1.5)
ax.set_ylabel('BTC Value (USDT)'); ax.set_title('BTC Position'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[2, 1])
ax.step(res['date'], res['btc_reinvest'], where='post', color='#f59e0b', linewidth=1.5)
ax.axhline(y=50, color='#f59e0b', linestyle=':', alpha=0.5, label='Cap: 50%')
ax.set_ylabel('BTC Reinvest %'); ax.set_ylim(0, 58); ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
# BNB value
ax = fig.add_subplot(gs[3, 0])
ax.fill_between(res['date'], 0, res['bnb_value'], color='#8b5cf6', alpha=0.3)
ax.plot(res['date'], res['bnb_value'], color='#7c3aed', linewidth=1.5)
ax.set_ylabel('BNB Value (USDT)'); ax.set_title('BNB Position'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[3, 1])
ax.step(res['date'], res['bnb_reinvest'], where='post', color='#8b5cf6', linewidth=1.5)
ax.axhline(y=40, color='#8b5cf6', linestyle=':', alpha=0.5, label='Cap: 40%')
ax.set_ylabel('BNB Reinvest %'); ax.set_ylim(0, 46); ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
# USDT + prices
ax = fig.add_subplot(gs[4, 0])
ax.fill_between(res['date'], 0, res['usdt_reserve'], color='#10b981', alpha=0.3)
ax.plot(res['date'], res['usdt_reserve'], color='#059669', linewidth=1.5)
ax.set_ylabel('USDT Reserve'); ax.set_title('Cash Reserve'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[4, 1])
ax.plot(res['date'], res['btc_close'], color='#f59e0b', linewidth=1, alpha=0.7, label='BTC')
axb = ax.twinx()
axb.plot(res['date'], res['bnb_close'], color='#8b5cf6', linewidth=1, alpha=0.7, label='BNB')
ax.set_ylabel('BTC (USDT)'); axb.set_ylabel('BNB (USDT)')
ax.set_title('Prices'); ax.grid(True, alpha=0.25)
ax.legend(loc='upper left', fontsize=8); axb.legend(loc='upper right', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-final.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-portfolio-final.png')
plt.show()

In [ ]:
# Chart 2: Stacked allocation
fig, ax = plt.subplots(figsize=(14, 7))
ax.stackplot(res['date'], res['btc_value'], res['bnb_value'], res['usdt_reserve'],
    labels=['BTC', 'BNB', 'USDT Reserve'], colors=['#f59e0b', '#8b5cf6', '#10b981'], alpha=0.8)
ax.plot(res['date'], res['total_injected'], color='gray', linewidth=1.5, linestyle='--', label='Total Injected')
ax.set_title('Portfolio Composition Over Time', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y')); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-final-stacked.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-portfolio-final-stacked.png')
plt.show()

In [ ]:
# Monthly cashflow
cashflow = []
for i, row in res.iterrows():
    btc_buy = 0; bnb_buy = 0; btc_sell = 0; bnb_sell = 0
    if 'BTC buy' in row['action']:
        import re
        m = re.search(r'BTC buy \$(\d+\.?\d*)', row['action'])
        if m: btc_buy = float(m.group(1))
    if 'BNB buy' in row['action']:
        m = re.search(r'BNB buy \$(\d+\.?\d*)', row['action'])
        if m: bnb_buy = float(m.group(1))
    if i > 0:
        btc_d = row['btc_held'] - res.iloc[i-1]['btc_held']
        if btc_d < -0.000001:
            btc_sell = abs(btc_d) * (row['btc_close'] + res.iloc[i-1]['btc_close']) / 2
        bnb_d = row['bnb_held'] - res.iloc[i-1]['bnb_held']
        if bnb_d < -0.000001:
            bnb_sell = abs(bnb_d) * (row['bnb_close'] + res.iloc[i-1]['bnb_close']) / 2
    cashflow.append({'date': row['date'], 'btc_buy': btc_buy, 'bnb_buy': bnb_buy,
        'btc_sell': btc_sell, 'bnb_sell': bnb_sell, 'reserve': row['usdt_reserve'],
        'portfolio': row['portfolio']})
cf = pd.DataFrame(cashflow)
print(f"{'Date':>10s} {'BTCbuy':>8s} {'BNBbuy':>8s} {'BTCsell':>8s} {'BNBsell':>8s} {'Resv':>8s} {'Portfolio':>10s}")
print('-'*66)
for _, r in cf.iterrows():
    print(f"{r['date'].strftime('%Y-%m'):>10s} {r['btc_buy']:>8.1f} {r['bnb_buy']:>8.1f} {r['btc_sell']:>8.1f} {r['bnb_sell']:>8.1f} {r['reserve']:>8.1f} {r['portfolio']:>10.1f}")
print()
print(f'Total BTC buys:  ${cf["btc_buy"].sum():.0f}')
print(f'Total BNB buys:  ${cf["bnb_buy"].sum():.0f}')
print(f'Total BTC sells: ${cf["btc_sell"].sum():.0f}')
print(f'Total BNB sells: ${cf["bnb_sell"].sum():.0f}')
print(f'Final reserve:   ${cf["reserve"].iloc[-1]:.0f}')
print(f'Final portfolio: ${cf["portfolio"].iloc[-1]:.0f}')